# 04 - Analysis

Cross-national comparisons (RQ1a, RQ1b) are descriptive; the campaign effects (H1a, H1b) are
tested with two-way ANOVAs. Diversity is the outlet-averaged weekly Simpson's Dz (one score per
country-week). Figures are separate for societal and political diversity and styled for APA7;
the ANOVA results are written as APA Word tables to `results/`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
import seaborn as sns
import statsmodels.formula.api as smf
import statsmodels.api as sm
from scipy import stats
import pingouin as pg
import helpers as h

h.set_apa_style()
FIG_DIR = h.FIGURES_DIR
FIG_DIR.mkdir(parents=True, exist_ok=True)

soc = pd.read_csv(h.RESULTS_DIR / "soc_div_country_week.csv")
pol = pd.read_csv(h.RESULTS_DIR / "pol_div_country_week.csv")

risk_pct = (pd.read_csv(h.SOCIAL_CLEAN, sep=h.CSV_SEP)
              .groupby("country")["Pol_Independence"].first().mul(100))
ORDER_BY_RISK = risk_pct.sort_values().index.tolist()
TIER_FILL = {"Low": "#d9d9d9", "Medium": "#969696", "High": "#525252"}
print("countries by ascending risk:", [(c, round(risk_pct[c])) for c in ORDER_BY_RISK])

## 1. Diversity by political-independence risk (RQ1a, RQ1b)

One figure per dimension: weekly Simpson's Dz per country, ordered by ascending
political-independence risk and shaded by risk tier. Descriptive, no across-tier test (tiers are
3 / 1 / 1 countries).

In [ ]:
def country_boxplot(df, dim_label, fname):
    fig, ax = plt.subplots(figsize=(5, 3.3))
    groups = [df.loc[df["country"] == cc, "simpson_Dz"].dropna().values for cc in ORDER_BY_RISK]
    bp = ax.boxplot(groups, showfliers=False, patch_artist=True, widths=0.6,
                    medianprops=dict(color="black"), boxprops=dict(color="black"),
                    whiskerprops=dict(color="black"), capprops=dict(color="black"))
    for patch, cc in zip(bp["boxes"], ORDER_BY_RISK):
        patch.set_facecolor(TIER_FILL[h.RISK_GROUP[cc]])
    for i, cc in enumerate(ORDER_BY_RISK, start=1):
        mu = np.nanmean(groups[i-1])
        ax.scatter(i, mu, marker="D", s=20, color="black", zorder=4)
        ax.annotate(f"{mu:.2f}", (i, mu), xytext=(8,0), textcoords="offset points",
                    va="top", ha="center", fontsize=8)
    ax.set_xticks(range(1, len(ORDER_BY_RISK) + 1), )
    ax.set_xticklabels([f"{h.COUNTRY_NAME[c]}\n({round(risk_pct[c])}%)" for c in ORDER_BY_RISK],
                       fontsize=9, va="top")
    ax.set_ylabel(f"Weekly Simpson's $D_z$ ({dim_label})"); ax.set_ylim(0, 1)
    ax.legend(handles=[Patch(facecolor=TIER_FILL[t], edgecolor="black", label=f"{t} risk")
                       for t in h.RISK_ORDER]
                      + [Line2D([0], [0], marker="D", color="w", markerfacecolor="black",
                                markersize=7, label="Mean")],
              loc="lower right", fontsize=8, ncol=2)
    fig.tight_layout(); fig.savefig(FIG_DIR / fname); plt.show()

country_boxplot(soc, "societal", "fig_rq1a_societal_by_country.png")
print("Figure 1. Societal Actor Diversity by Country, Ordered by Political-Independence Risk.")
country_boxplot(pol, "political", "fig_rq1b_political_by_country.png")
print("Figure 2. Political Actor Diversity (Government vs Opposition) by Country.")

## Testing whether differences are significant across countries

### Assumptions

In [ ]:
# Normality: mostly normally distributed
for c in soc['country'].unique():
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    subset = soc[soc['country'] == c]
    pg.qqplot(subset['simpson_Dz'], ax=axes[0])
    axes[0].set_title(f'Q-Q Plot for {c}', fontweight='bold')
    sns.histplot(subset['simpson_Dz'], kde=True, ax=axes[1])
    axes[1].set_title(f'Histogram for {c}', fontweight='bold')
    plt.tight_layout()
    plt.show()

In [ ]:
soc.head()

In [ ]:
pol.head()

In [ ]:
# Normality: mostly normally distributed
for c in pol['country'].unique():
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    subset = pol[pol['country'] == c]
    pg.qqplot(subset['simpson_Dz'], ax=axes[0])
    axes[0].set_title(f'Q-Q Plot for {c}', fontweight='bold')
    sns.histplot(subset['simpson_Dz'], kde=True, ax=axes[1])
    axes[1].set_title(f'Histogram for {c}', fontweight='bold')
    plt.tight_layout()
    plt.show()

In [ ]:
pg.homoscedasticity(soc, dv='simpson_Dz', group='country') #violates homoscedasticity assumption, so we will use Welch's ANOVA

In [ ]:
pg.homoscedasticity(pol, dv='simpson_Dz', group='country') #violates homoscedasticity assumption, so we will use Welch's ANOVA

In [ ]:
soc.groupby("country")["simpson_Dz"].std()

s = soc.groupby("country")["simpson_Dz"].std()
(s.max() / s.min()) ** 2   # ratio of largest to smallest variance

In [ ]:
pol.groupby("country")["simpson_Dz"].std()

s = pol.groupby("country")["simpson_Dz"].std()
(s.max() / s.min()) ** 2   # ratio of largest to smallest variance

In [ ]:
pg.welch_anova(dv='simpson_Dz', 
               between='country', 
               data=soc).round(3)


In [ ]:
pg.welch_anova(dv='simpson_Dz', 
               between='country', 
               data=pol).round(3)

In [ ]:
soc_table_country_pairwise = pg.pairwise_gameshowell(dv='simpson_Dz', 
                   between='country', 
                   data=soc).round(3)
h.apa_table(soc_table_country_pairwise, number=6,
            title="Pairwise Comparison of Societal Actor Diversity by Country",
            path=h.TABLES_DIR / "Table6_societal_pairwisecountry.docx", index_label="Country",
            note="Games-Howell post-hoc comparisons.")
soc_table_country_pairwise.to_csv(h.TABLES_DIR / "Table6_societal_pairwise.csv", index=False)

soc_table_country_pairwise

In [ ]:
pol_table_country_pairwise = pg.pairwise_gameshowell(dv='simpson_Dz', 
                   between='country', 
                   data=pol).round(3)
h.apa_table(pol_table_country_pairwise, number=8,
            title="Pairwise Comparison of Political Actor Diversity by Country",
            path=h.TABLES_DIR / "Table8_political_pairwisecountry.docx", index_label="Country",
            note="Games-Howell post-hoc comparisons.")
pol_table_country_pairwise

## Campaign Period

In [ ]:
SRC = {"C(campaign_period)": "Campaign period", "C(country)": "Country",
       "C(campaign_period):C(country)": "Campaign \u00d7 Country", "Residual": "Error"}

def anova_rows(df, dim):
    m = smf.ols("simpson_Dz ~ C(campaign_period) * C(country)", data=df).fit()
    a = sm.stats.anova_lm(m, typ=2, robust='hc3'); ss_res = a.loc["Residual", "sum_sq"]
    rows = []
    for src in ["C(campaign_period)", "C(country)", "C(campaign_period):C(country)", "Residual"]:
        r = a.loc[src]; resid = src == "Residual"
        eta = "" if resid else f"{r['sum_sq']/(r['sum_sq']+ss_res):.3f}".lstrip("0")
        rows.append({"Dimension": dim, "Source": SRC[src], "df": int(r["df"]),
                     "SS": f"{r['sum_sq']:.3f}", "F": "" if resid else f"{r['F']:.2f}",
                     "p": "" if resid else h.apa_p(r["PR(>F)"]), "\u03b7\u00b2p": eta})
    return rows

anova_tbl = pd.DataFrame(anova_rows(soc, "Societal") + anova_rows(pol, "Political"))
print(anova_tbl.to_string(index=False))

h.apa_table(anova_tbl, number=6,
            title="Two-Way ANOVAs of Weekly Actor Diversity by Campaign Period and Country",
            path=h.TABLES_DIR / "Table6_campaign_anova.docx",
            note=("Dz = standardised Simpson's diversity index, computed per outlet-week and "
                  "averaged across outlets. \u03b7\u00b2p = partial eta-squared. Weekly scores "
                  "are autocorrelated, so p values are approximate."))
anova_tbl.to_csv(h.TABLES_DIR / "Table7_campaign_anova.csv", index=False)
print("\nsaved Table7_campaign_anova.docx / .csv")

In [ ]:
refl = pd.read_csv(h.RESULTS_DIR / "pol_reflective_gov_share.csv").set_index("country")["reflective_gov_share"]
sd = soc.groupby("country")["simpson_Dz"].mean()
pd_ = pol.groupby("country")["simpson_Dz"].mean()
polg = pol.groupby("country")["p_gov"].mean()

bench = pd.DataFrame({
    "Risk %": risk_pct.round().astype(int),
    "Societal Dz": sd.round(3), "Open shortfall societal": (1 - sd).round(3),
    "Political Dz": pd_.round(3), "Gov share %": (polg * 100).round(1), "Open shortfall political": (1 - pd_).round(3),
    "Seat share %": (refl * 100).round(1), "Reflective dev (pp)": ((polg - refl) * 100).round(1),
}).reindex(h.COUNTRY_ORDER)
bench.index = [h.COUNTRY_NAME[c] for c in bench.index]
print(bench.to_string())

h.apa_table(bench, number=5,
            title="Actor Diversity Against the Open and Reflective Benchmarks, by Country",
            path=h.TABLES_DIR / "Table5_benchmarks.docx", index_label="Country",
            note=("Open shortfall = 1 - societal Dz (distance from equal four-category "
                  "representation). Reflective dev = governing-party share of political actors "
                  "minus their share of parliamentary seats, in percentage points."))
bench.to_csv(h.TABLES_DIR / "Table5_benchmarks.csv")

### Campaign vs routine figures (separate per dimension)

In [ ]:
def campaign_dumbbell(df, dim_label, fname):
    fig, ax = plt.subplots(figsize=(5, 3.3))
    means = (df.groupby(["country", "campaign_period"])["simpson_Dz"].mean()
                .unstack("campaign_period").reindex(h.COUNTRY_ORDER))

    def stars(cc):                                    # per-country Welch test (as in Table 7)
        d = df[df["country"] == cc]
        a = d.loc[d["campaign_period"] == 0, "simpson_Dz"].dropna()
        b = d.loc[d["campaign_period"] == 1, "simpson_Dz"].dropna()
        p = stats.ttest_ind(b, a, equal_var=False).pvalue
        return "***" if p < .001 else "**" if p < .01 else "*" if p < .05 else ""

    y = np.arange(len(means))
    for i, cc in enumerate(means.index):
        x0, x1 = means.loc[cc, 0], means.loc[cc, 1]
        ax.plot([x0, x1], [i, i], color="0.6", lw=1.5, zorder=1)
        ax.scatter(x0, i, facecolor="white", edgecolor="black", s=55, zorder=2, label="Routine" if i == 0 else "")
        ax.scatter(x1, i, color="black", s=55, zorder=2, label="Campaign" if i == 0 else "")
        lo, hi = (x0, x1) if x0 <= x1 else (x1, x0)   # label by left/right, not by period
        ax.annotate(f"{lo:.2f}", (lo, i), xytext=(-15, 5), textcoords="offset points", ha="center", fontsize=7)
        ax.annotate(f"{hi:.2f}", (hi, i), xytext=(15, 5), textcoords="offset points", ha="center", fontsize=7)
    ax.set_yticks(y)
    ax.set_yticklabels([f"{h.COUNTRY_NAME[c]} {stars(c)}".strip() for c in means.index])
    ax.set_xlim(0, 1); ax.set_xlabel(f"Mean weekly Simpson's $D_z$ ({dim_label})")
    ax.legend(loc="lower left", fontsize=9)
    fig.tight_layout(); fig.savefig(FIG_DIR / fname); plt.show()

campaign_dumbbell(soc, "societal", "fig_h1a_societal_campaign.png")
print("Figure 3. Societal Actor Diversity in Campaign Versus Routine Periods, by Country.")
campaign_dumbbell(pol, "political", "fig_h1b_political_campaign.png")
print("Figure 4. Political Actor Diversity in Campaign Versus Routine Periods, by Country.")

## 3. Diversity against the benchmarks (RQ1a, RQ1b)

Country means against the normative benchmarks: societal diversity vs open (shortfall = 1 - Dz),
and the government-opposition balance vs open (Dz, gov share) and reflective (gov share minus
seat share).

In [ ]:
refl = pd.read_csv(h.RESULTS_DIR / "pol_reflective_gov_share.csv").set_index("country")["reflective_gov_share"]
sd = soc.groupby("country")["simpson_Dz"].mean()
pd_ = pol.groupby("country")["simpson_Dz"].mean()
polg = pol.groupby("country")["p_gov"].mean()

bench = pd.DataFrame({
    "Risk %": risk_pct.round().astype(int),
    "Societal Dz": sd.round(3), "Open shortfall societal": (1 - sd).round(3),
    "Political Dz": pd_.round(3), "Gov share %": (polg * 100).round(1), "Open shortfall political": (1 - pd_).round(3),
    "Seat share %": (refl * 100).round(1), "Reflective dev (pp)": ((polg - refl) * 100).round(1),
}).reindex(h.COUNTRY_ORDER)
bench.index = [h.COUNTRY_NAME[c] for c in bench.index]
print(bench.to_string())

h.apa_table(bench, number=5,
            title="Actor Diversity Against the Open and Reflective Benchmarks, by Country",
            path=h.TABLES_DIR / "Table5_benchmarks.docx", index_label="Country",
            note=("Open shortfall = 1 - societal Dz (distance from equal four-category "
                  "representation). Reflective dev = governing-party share of political actors "
                  "minus their share of parliamentary seats, in percentage points."))
bench.to_csv(h.TABLES_DIR / "Table5_benchmarks.csv")

# Outlet-level Analyses

## 4. Public broadcasters vs newspapers (RQ2)

Outlet-week scores grouped into public service broadcasters and (privately owned) newspapers,
per country. Descriptive.

In [ ]:
soc_pn = pd.read_csv(h.RESULTS_DIR / "soc_div_psb_news_paired.csv")
pol_pn = pd.read_csv(h.RESULTS_DIR / "pol_div_psb_news_paired.csv")

In [ ]:
soc_pn.head()

In [ ]:
pg.homoscedasticity(soc_pn, dv='simpson_Dz', group='country') #violates homoscedasticity assumption, so we will use Welch's ANOVA

In [ ]:
# Normality: mostly normally distributed
for c in soc_pn['country'].unique():
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    subset = soc_pn[soc_pn['country'] == c]
    pg.qqplot(subset['simpson_Dz'], ax=axes[0])
    axes[0].set_title(f'Q-Q Plot for {c}', fontweight='bold')
    sns.histplot(subset['simpson_Dz'], kde=True, ax=axes[1])
    axes[1].set_title(f'Histogram for {c}', fontweight='bold')
    plt.tight_layout()
    plt.show()

In [ ]:
pg.homoscedasticity(pol_pn, dv='simpson_Dz', group='country') #violates homoscedasticity assumption, so we will use Welch's ANOVA

In [ ]:
# Normality: mostly normally distributed
for c in pol_pn['country'].unique():
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    subset = pol_pn[pol_pn['country'] == c]
    pg.qqplot(subset['simpson_Dz'], ax=axes[0])
    axes[0].set_title(f'Q-Q Plot for {c}', fontweight='bold')
    sns.histplot(subset['simpson_Dz'], kde=True, ax=axes[1])
    axes[1].set_title(f'Histogram for {c}', fontweight='bold')
    plt.tight_layout()
    plt.show()

In [ ]:
soc_pn.head()

In [ ]:
# --- PSB vs newspaper, respecting the within-week pairing --------------------
# Each country-week contributes one PSB and one newspaper score that share the
# same week's events, so the two rows are NOT independent. Differencing within
# the pair collapses them to a single gap per country-week; those gaps ARE
# independent across weeks, which is the assumption the t test actually needs.

def psb_gap(df):
    wide = (df.pivot_table(index=["country", "year_week"],
                           columns="group", values="simpson_Dz")
              .dropna())                                   # complete PSB+news pairs only
    wide["gap"] = wide["Public broadcaster"] - wide["Newspaper"]
    return wide.reset_index()

def psb_rows(df, dim):
    g = psb_gap(df)
    rows = []
    for cc in h.COUNTRY_ORDER:
        d = g.loc[g["country"] == cc, "gap"].dropna()
        r = pg.ttest(d, 0).iloc[0]
        lo, hi = r["CI95%"]
        n = len(d)
        hedges = r["cohen-d"] * (1 - 3 / (4 * (n - 1) - 1))   # small-sample bias correction
        rows.append({"Dimension": dim, "Country": h.COUNTRY_NAME[cc],
                     "PSB - News": f"{d.mean():+.3f}", "n weeks": n,
                     "95% CI": f"[{lo:+.3f}, {hi:+.3f}]",
                     "t": f"{r['T']:.2f}", "p": h.apa_p(r["p-val"]),
                     "hedges": f"{hedges:.2f}"})
    return rows

In [ ]:
# per-country simple effects, both dimensions stacked (as in the campaign table)
psb_tbl = pd.DataFrame(psb_rows(soc_pn, "Societal") + psb_rows(pol_pn, "Political"))
print(psb_tbl.to_string(index=False))

h.apa_table(psb_tbl, number=9,
            title="Public Broadcaster Versus Newspaper Actor Diversity, by Country",
            path=h.TABLES_DIR / "Table9_psb_news.docx",
            note=("PSB - News = mean weekly difference in Simpson's Dz (public broadcaster minus "
                  "newspaper), tested per country with a one-sample t test on the country-week "
                  "gaps. d_z = paired Hedges' g. Positive values indicate higher diversity on the "
                  "public broadcaster."))
psb_tbl.to_csv(h.TABLES_DIR / "Table9_psb_news.csv", index=False)

In [ ]:
# --- Omnibus tests on the gaps, both dimensions -----------------------------
def gap_omnibus(df, dim):
    g = psb_gap(df)

    # Interaction: does the gap vary across countries? Welch one-way, as for the
    # country tests earlier (robust to the unequal gap variances across countries).
    inter = pg.welch_anova(dv="gap", between="country", data=g).iloc[0]

    # Main effect: average PSB-newspaper gap holding country constant. Sum coding
    # makes the intercept the (unweighted) grand mean of the country means, so the
    # intercept's t test is the country-adjusted "is the gap != 0 overall". HC3
    # keeps the SE robust to the same heteroscedasticity Welch handles above.
    main_adj = smf.ols("gap ~ C(country, Sum)", data=g).fit(cov_type="HC3")
    b0 = main_adj.params["Intercept"]

    return {"Dimension": dim,
            "Overall PSB - News": f"{b0:+.3f}",
            "Main: t":  f"{main_adj.tvalues['Intercept']:.2f}",
            "Main: df": int(main_adj.df_resid),
            "Main: p":  h.apa_p(main_adj.pvalues["Intercept"]),
            "Inter: F":  f"{inter['F']:.2f}",
            "Inter: df": f"{int(inter['ddof1'])}, {inter['ddof2']:.1f}",
            "Inter: p":  h.apa_p(inter["p-unc"]),
            "Inter: np2": f"{inter['np2']:.3f}".lstrip("0")}

omni = pd.DataFrame([gap_omnibus(soc_pn, "Societal"), gap_omnibus(pol_pn, "Political")])
print("\n", omni.to_string(index=False))

h.apa_table(omni, number=10,
            title="Overall and Cross-National Tests of the PSB-Newspaper Diversity Gap",
            path=h.TABLES_DIR / "Table10_psb_omnibus.docx",
            note=("Main effect = average PSB - newspaper gap holding country constant (intercept "
                  "of gap ~ country with sum coding, HC3-robust SE), df = error. Interaction = "
                  "Welch's one-way ANOVA of the gap across countries; np2 = partial eta-squared. "
                  "Positive values indicate higher diversity on the public broadcaster."))
omni.to_csv(h.TABLES_DIR / "Table10_psb_omnibus.csv", index=False)


In [ ]:
def psb_dumbbell(df, dim_label, fname):
    fig, ax = plt.subplots(figsize=(5, 3.3))
    means = (df.groupby(["country", "group"])["simpson_Dz"].mean()
                .unstack("group").reindex(h.COUNTRY_ORDER))
    gaps = psb_gap(df)                                 # reuse the paired-gap builder

    def stars(cc):                                     # per-country PAIRED test (as in Table 9)
        d = gaps.loc[gaps["country"] == cc, "gap"].dropna()
        p = pg.ttest(d, 0).iloc[0]["p-val"]
        return "***" if p < .001 else "**" if p < .01 else "*" if p < .05 else ""

    y = np.arange(len(means))
    for i, cc in enumerate(means.index):
        x_news, x_psb = means.loc[cc, "Newspaper"], means.loc[cc, "Public broadcaster"]
        ax.plot([x_news, x_psb], [i, i], color="0.6", lw=1.5, zorder=1)
        ax.scatter(x_news, i, facecolor="white", edgecolor="black", s=55, zorder=2,
                   label="Newspaper" if i == 0 else "")
        ax.scatter(x_psb, i, color="black", s=55, zorder=2,
                   label="Public broadcaster" if i == 0 else "")
        lo, hi = sorted((x_news, x_psb))               # label by left/right, not by outlet type
        ax.annotate(f"{lo:.2f}", (lo, i), xytext=(-10, 5), textcoords="offset points", ha="center", fontsize=7)
        ax.annotate(f"{hi:.2f}", (hi, i), xytext=(10, 5), textcoords="offset points", ha="center", fontsize=7)
    ax.set_yticks(y)
    ax.set_yticklabels([f"{h.COUNTRY_NAME[c]} {stars(c)}".strip() for c in means.index])
    ax.set_xlim(0, 1); ax.set_xlabel(f"Mean weekly Simpson's $D_z$ ({dim_label})")
    ax.legend(loc="lower left", fontsize=9)
    fig.tight_layout(); fig.savefig(FIG_DIR / fname); plt.show()

psb_dumbbell(soc_pn, "societal", "fig_rq2_societal_psb.png")
print("Figure 5. Societal Actor Diversity in Public Broadcasters Versus Newspapers, by Country.")
psb_dumbbell(pol_pn, "political", "fig_rq2_political_psb.png")
print("Figure 6. Political Actor Diversity in Public Broadcasters Versus Newspapers, by Country.")


## 5. Issue differences (RQ3)

Diversity by issue (immigration vs inflation), per country, on the outlet-week-topic scores, with
an exploratory two-way ANOVA `Dz ~ issue * country`.

In [ ]:
sct = pd.read_csv(h.RESULTS_DIR / "soc_div_country_week_topic.csv")
pct = pd.read_csv(h.RESULTS_DIR / "pol_div_country_week_topic.csv")

rows = []
for cc in h.COUNTRY_ORDER:
    r = {"Country": h.COUNTRY_NAME[cc]}
    for d, lab in [(sct, "Societal"), (pct, "Political")]:
        g = d[d["country"] == cc].groupby("topic_label")["simpson_Dz"].mean()
        r[f"{lab}: immig."] = round(g.get("immigration", np.nan), 3)
        r[f"{lab}: infl."]  = round(g.get("inflation", np.nan), 3)
    rows.append(r)
rq4 = pd.DataFrame(rows).set_index("Country")
print(rq4.to_string())
h.apa_table(rq4, number=11, title="Actor Diversity by Issue, by Country",
            path=h.TABLES_DIR / "Table11_issue.docx", index_label="Country",
            note="Mean weekly Simpson's Dz over outlet-week-issue scores.")
rq4.to_csv(h.TABLES_DIR / "Table11_issue.csv")

In [ ]:
# --- RQ4 figures: issue contrast per country (parallel to campaign / PSB) -----
# Input is the country-week-topic, outlet-averaged data (same grain as the ANOVA).
# Immigration and inflation can occur in the SAME country-week, so they are paired:
# the stars come from a one-sample t on the within-week gap, exactly as in RQ2.

def issue_gap(df):
    wide = (df.pivot_table(index=["country", "year_week"],
                            columns="topic_label", values="simpson_Dz")
            .dropna())                                 # keep weeks with BOTH issues
    wide["gap"] = wide["inflation"] - wide["immigration"]
    return wide.reset_index()

def issue_dumbbell(df, dim_label, fname):
    fig, ax = plt.subplots(figsize=(5, 3.3))
    means = (df.groupby(["country", "topic_label"])["simpson_Dz"].mean()
                .unstack("topic_label").reindex(h.COUNTRY_ORDER))
    gaps = issue_gap(df)                                 # reuse the paired-gap builder

    def stars(cc):                                       # per-country PAIRED test
        d = gaps.loc[gaps["country"] == cc, "gap"].dropna()
        p = pg.ttest(d, 0).iloc[0]["p-val"]
        return "***" if p < .001 else "**" if p < .01 else "*" if p < .05 else ""

    y = np.arange(len(means))
    for i, cc in enumerate(means.index):
        x_imm, x_inf = means.loc[cc, "immigration"], means.loc[cc, "inflation"]
        ax.plot([x_imm, x_inf], [i, i], color="0.6", lw=1.5, zorder=1)
        ax.scatter(x_imm, i, facecolor="white", edgecolor="black", s=55, zorder=2,
                    label="Immigration" if i == 0 else "")
        ax.scatter(x_inf, i, color="black", s=55, zorder=2,
                    label="Inflation" if i == 0 else "")
        lo, hi = sorted((x_imm, x_inf))                  # label by left/right, not by issue
        ax.annotate(f"{lo:.2f}", (lo, i), xytext=(-10, 5), textcoords="offset points", ha="center", fontsize=7)
        ax.annotate(f"{hi:.2f}", (hi, i), xytext=(10, 5), textcoords="offset points", ha="center", fontsize=7)
    ax.set_yticks(y)
    ax.set_yticklabels([f"{h.COUNTRY_NAME[c]} {stars(c)}".strip() for c in means.index])
    ax.set_xlim(0, 1); ax.set_xlabel(f"Mean weekly Simpson's $D_z$ ({dim_label})")
    ax.legend(loc="lower left", fontsize=9)
    fig.tight_layout(); fig.savefig(FIG_DIR / fname); plt.show()

issue_dumbbell(sct, "societal", "fig_rq4_societal_issue.png")
print("Figure 7. Societal Actor Diversity in Immigration Versus Inflation Coverage, by Country.")
issue_dumbbell(pct, "political", "fig_rq4_political_issue.png")
print("Figure 8. Political Actor Diversity in Immigration Versus Inflation Coverage, by Country.")

In [ ]:
SRC4 = {"C(topic_label)": "Issue", "C(country)": "Country",
        "C(topic_label):C(country)": "Issue \u00d7 Country", "Residual": "Error"}
sct = pd.read_csv(h.RESULTS_DIR / "soc_div_country_week_topic.csv")
pct = pd.read_csv(h.RESULTS_DIR / "pol_div_country_week_topic.csv")

def topic_anova_rows(d, dim):
    m = smf.ols("simpson_Dz ~ C(topic_label) * C(country)", data=d).fit()
    a = sm.stats.anova_lm(m, typ=2, robust='hc3'); ss_res = a.loc["Residual", "sum_sq"]
    out = []
    for src in ["C(topic_label)", "C(country)", "C(topic_label):C(country)", "Residual"]:
        r = a.loc[src]; resid = src == "Residual"
        eta = "" if resid else f"{r['sum_sq']/(r['sum_sq']+ss_res):.3f}".lstrip("0")
        out.append({"Dimension": dim, "Source": SRC4[src], "df": int(r["df"]), "SS": f"{r['sum_sq']:.3f}",
                    "F": "" if resid else f"{r['F']:.2f}", "p": "" if resid else h.apa_p(r["PR(>F)"]),
                    "\u03b7\u00b2p": eta})
    return out

rq4_aov = pd.DataFrame(topic_anova_rows(sct, "Societal") + topic_anova_rows(pct, "Political"))
print("\n", rq4_aov.to_string(index=False))
h.apa_table(rq4_aov, number=13,
            title="Two-Way ANOVAs of Weekly Actor Diversity by Issue and Country",
            path=h.TABLES_DIR / "Table13_issue_anova.docx",
            note=("Computed on country-week-issue scores (outlet-averaged), as for the campaign "
                  "ANOVAs. \u03b7\u00b2p = partial eta-squared. Weekly scores are autocorrelated, "
                  "so p values are approximate."))
rq4_aov.to_csv(h.TABLES_DIR / "Table13_issue_anova.csv", index=False)

In [ ]:
soc_table_country_pairwise = pg.pairwise_gameshowell(dv='simpson_Dz', 
                   between='topic_label', 
                   data=sct).round(3)
# h.apa_table(soc_table_country_pairwise, number=6,
#             title="Pairwise Comparison of Societal Actor Diversity by Country",
#             path=h.TABLES_DIR / "Table6_societal_pairwisecountry.docx", index_label="Country",
#             note="Games-Howell post-hoc comparisons.")
# soc_table_country_pairwise.to_csv(h.TABLES_DIR / "Table6_societal_pairwise.csv", index=False)

soc_table_country_pairwise

In [ ]:
def issue_gap(df):                                   # df = country-week-topic, outlet-averaged
    w = (df.pivot_table(index=["country", "year_week"],
                        columns="topic_label", values="simpson_Dz").dropna())
    w["gap"] = w["inflation"] - w["immigration"]      # +ve = inflation more diverse
    return w.reset_index()

def issue_rows(df, dim):
    g = issue_gap(df)
    rows = []
    for cc in h.COUNTRY_ORDER:
        d = g.loc[g["country"] == cc, "gap"].dropna()
        r = pg.ttest(d, 0).iloc[0]
        lo, hi = r["CI95%"]; n = len(d)
        hedges = r["cohen-d"] * (1 - 3 / (4 * (n - 1) - 1))   # paired g (d_z), bias-corrected
        rows.append({"Dimension": dim, "Country": h.COUNTRY_NAME[cc],
                    "Inflation - immigration": f"{d.mean():+.3f}", "n weeks": n,
                    "95% CI": f"[{lo:+.3f}, {hi:+.3f}]",
                    "t": f"{r['T']:.2f}", "p": h.apa_p(r["p-val"]), "hedges": f"{hedges:.2f}"})
    return rows

issue_tbl = pd.DataFrame(issue_rows(sct, "Societal") + issue_rows(pct, "Political"))
print(issue_tbl.to_string(index=False))
h.apa_table(issue_tbl, number=14,
            title="Issue Differences in Weekly Actor Diversity, by Country",
            path=h.TABLES_DIR / "Table14_issue_contrasts.docx",
            note=("Inflation - immigration = mean within-week difference in Simpson's Dz, tested per "
                "country with a one-sample t test on the country-week gaps (simple-effects "
                "decomposition of the Issue x Country interaction). hedges = paired Hedges' g. "
                "Positive values indicate higher diversity in inflation coverage."))